In [22]:
from skmultilearn.model_selection import iterative_train_test_split
import pandas as pd
import numpy as np
from config import images_path
from pathlib import Path
import os, json

In [23]:
to_predict = (
    'female', 'male',
    'unicorn', 'pegasus', 'earth pony', 'alicorn',
    'simple background', 'monochrome',
    'clothes', 'wings', 'horn', 'chest fluff', 'ear fluff', 'hat', 'jewelry', 'food', 'foal',
    'looking at you', 'smiling', 'open mouth', 'blushing', 'sitting', 'raised hoof', 'eyes closed',
    'twilight sparkle', 'fluttershy', 'rainbow dash', 'pinkie pie', 'rarity', 'applejack'
)

In [24]:
category_map = {
    'pony_type': ['unicorn', 'pegasus', 'earth pony', 'alicorn'],
    'features': ['female', 'male', 'wings', 'horn', 'chest fluff', 'ear fluff', 'foal'],
    'appearance': ['simple background', 'monochrome', 'clothes', 'hat', 'jewelry', 'food'],
    'pose_expression': ['looking at you', 'smiling', 'open mouth', 'blushing', 'sitting', 'raised hoof', 'eyes closed'],
    'character': ['twilight sparkle', 'fluttershy', 'rainbow dash', 'pinkie pie', 'rarity', 'applejack']
}

In [25]:
df = pd.read_csv('data/retagged.csv')
df['tags'] = df['tags'].str.split('|')

In [ ]:
def to_annotation(row):
    return {
        "data": {
            "image": f'/data/local-files/?d=pone-test-split/{row['id']}.{row['image_format']}'
        },
        "predictions": [{
            "model_version": "preannotations",
            "result": [
                {
                    "from_name": category,
                    "to_name": "image",
                    "type": "choices",
                    "value": {
                        "choices": choices
                    }
                }
                for category, tags in category_map.items()
                    if (choices := list(set(row['tags']).intersection(tags)))
            ]
        }]
    }

In [27]:
annotations = [to_annotation(row) for _, row in df.iterrows()]

Uploading the dataset shards:   6%|▋         | 2/31 [08:39<2:05:25, 259.52s/ shards]


In [28]:
with open('data/ls_annotations.json', 'w') as output:
    json.dump(annotations, output)

In [29]:
def multi_hot(tags: list, tag_names: list):
    tags = np.array([tag_names.index(tag) for tag in tags if tag in tag_names])
    encoding = np.zeros(len(tag_names), dtype=np.uint8)

    if len(tags):
        encoding[tags] = 1

    return encoding

def stratified_split(tag_col: pd.Series, p_val, p_test):
    y = np.stack(tag_col.to_numpy())

    if os.path.exists('data/indices.npz'):
        with np.load('data/indices.npz') as stored:
            idx_train = stored['train']
            idx_val = stored['val']
            idx_test = stored['test']

        y_train = y[idx_train]
        y_val = y[idx_val]
        y_test = y[idx_test]
    else:
        idx = tag_col.index.to_numpy().reshape(-1, 1)

        idx_train, y_train, idx_test, y_test = iterative_train_test_split(idx, y, p_test)
        idx_train, y_train, idx_val, y_val = iterative_train_test_split(idx_train, y_train, p_val / (1 - p_test))

        idx_train = idx_train.reshape(-1)
        idx_val = idx_val.reshape(-1)
        idx_test = idx_test.reshape(-1)

        np.savez(
            'data/indices.npz',
            train=idx_train,
            val=idx_val,
            test=idx_test
        )
    
    return idx_train, y_train, idx_val, y_val, idx_test, y_test

In [30]:
out_path = Path('/home/brambles/Documents/A-Py/V1/data/out')

df['tags'] = df['tags'].map(lambda tags: multi_hot(tags, to_predict))
df['image'] = ''
df.head()

,id,created_at,image_format,score,tags,image
0,1,2012-01-02 03:12:33.000,png,3003,"[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",
1,5,2012-01-02 03:47:25.000,png,1975,"[1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, ...",
2,9,2012-01-02 04:13:40.000,png,746,"[1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, ...",
3,10,2012-01-02 04:14:24.000,png,841,"[1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, ...",
4,13,2012-01-02 04:19:04.000,png,719,"[1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, ...",


In [31]:
idx_train, y_train, idx_val, y_val, idx_test, y_test = stratified_split(df['tags'], 0.05, 0.05)

In [32]:
y_train.shape, len(idx_val), len(idx_test)

((174407, 30), 10593, 11929)

In [ ]:
for split_idxs, split_name in ((idx_train, 'train'), (idx_val, 'validation'), (idx_test, 'test')):
    output_path = out_path / split_name

    if not os.path.exists(output_path):
        os.mkdir(output_path)

    for idx in split_idxs:
        img_id, img_format, tags = df.iloc[idx][['id', 'image_format', 'tags']]
        file_name = f'{img_id}.{img_format}'
        out_file_path = output_path / file_name

        if not os.path.exists(out_file_path):
            os.symlink(images_path / file_name, out_file_path)

In [34]:
from datasets import Dataset
from config import hf_token
from datasets.features import Value, ClassLabel, Features, List
import datasets

In [36]:
df.loc[idx_train, 'image'] = df.apply(lambda row: str(out_path / 'train' / f'{row['id']}.{row['image_format']}'), axis=1)
df.loc[idx_val, 'image'] = df.apply(lambda row: str(out_path / 'validation' / f'{row['id']}.{row['image_format']}'), axis=1)
df.loc[idx_test, 'image'] = df.apply(lambda row: str(out_path / 'test' / f'{row['id']}.{row['image_format']}'), axis=1)

features = Features({
    'id': Value('int32'),
    'image_format': Value('string'),
    'tags': List(ClassLabel(names=to_predict)),
    'image': datasets.features.Image()
})

ds_train, ds_val, ds_test = [
    Dataset.from_pandas(
        df.iloc[idx][['id', 'image_format', 'tags', 'image']],
        features=features,
        split=split_name,
        preserve_index=False
    )
    for idx, split_name in ((idx_train, 'train'), (idx_val, 'validation'), (idx_test, 'test')) 
]

ds = datasets.DatasetDict({
    'train': ds_train,
    'validation': ds_val,
    'test': ds_test
})

ds.push_to_hub('Brambles/Ponies', private=True, max_shard_size='1GB', token=hf_token)


Uploading the dataset shards:   0%|          | 0/31 [00:00<?, ? shards/s]










Map: 100%|██████████| 5627/5627 [00:01<00:00, 3722.65 examples/s]




Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 25.71ba/s]
Processing Files (1 / 1): 100%|██████████|  929MB /  929MB, 69.6MB/s  
New Data Upload: 100%|██████████| 4.54MB / 4.54MB,  374kB/s  
Uploading the dataset shards:   3%|▎         | 1/31 [00:05<02:56,  5.88s/ shards]










Map: 100%|██████████| 5626/5626 [00:01<00:00, 3512.02 examples/s]




Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 26.22ba/s]
Processing Files (1 / 1): 100%|██████████|  910MB /  910MB, 69.9MB/s  
New Data Upload: 100%|██████████|  656kB /  656kB, 54.7kB/s  
Uploading the dataset shards:   6%|▋         | 2/31 [00:11<02:47,  5.77s/ shards]










Map: 100%|██████████| 5626/5626 [00:01<00:00, 3643.63 examples/s]




Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 25.97ba/s]
Proc

CommitInfo(commit_url='https://huggingface.co/datasets/Brambles/Ponies/commit/53491aa3dc8944faa911926db0a9a9ae3b448eb4', commit_message='Upload dataset', commit_description='', oid='53491aa3dc8944faa911926db0a9a9ae3b448eb4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Brambles/Ponies', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Brambles/Ponies'), pr_revision=None, pr_num=None)